In [2]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import numpy as np 
import pandas as pd 

device = torch.device('cuda' if torch.cuda.is_available else 'cpu')

device

device(type='cuda')

In [ ]:
class simpleLSTMwEmbed(nn.Module) :

    def __init__(self, embedding_dim, vocab_size, hidden_dim, device, embedding_weights = None, return_sequence_states = False) :
        super().__init__()
        self.hidden_dim = hidden_dim
        self.return_sequence_states = return_sequence_states
        self.device = device

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        if embedding_weights!= None : 
            with torch.no_grad() :
                self.embedding.weight.copy_(embedding_weights)

        self.forget_gate = nn.Linear(
            in_features = embedding_dim + hidden_dim,
            out_features = hidden_dim 
        )
        
        self.input_gate_i = nn.Linear(
            in_features = embedding_dim + hidden_dim,
            out_features = hidden_dim 
        )

        self.input_gate_c = nn.Linear(
            in_features = embedding_dim + hidden_dim,
            out_features = hidden_dim 
        )

        self.output_gate = nn.Linear(
            in_features = embedding_dim + hidden_dim,
            out_features = hidden_dim 
        )


    def forward(self, input_batch) : 
        # input shape : (B, T)
        batch_size, timesteps_count = input_batch.shape    
        embedded_batch = self.embedding(input_batch) # shaoe = (B,T,E) , E => embedding dimension


        hidden_state = torch.zeros(size=(batch_size, self.hidden_dim), device= self.device)
        cell_state = torch.zeros(size=(batch_size, self.hidden_dim), device= self.device)

        if self.return_sequence_states :
            hidden_states = torch.zeros(size=(batch_size, timesteps_count, self.hidden_dim), device= self.device)
            cell_states = torch.zeros(size=(batch_size, timesteps_count, self.hidden_dim), device= self.device)

        for timestep in range(timesteps_count):
            input = embedded_batch[: , timestep, :]    # shape = (B, E)
            X_t = torch.cat(tensors=(hidden_state,input ), dim=1)

            Ft = torch.sigmoid(self.forget_gate(X_t))
            cell_state = cell_state * Ft

            It = torch.sigmoid(self.input_gate_i(X_t))
            Canditate_t = torch.tanh(self.input_gate_c(X_t))
            cell_state = cell_state + (It * Canditate_t)

            Ot = torch.sigmoid(self.output_gate(X_t))
            hidden_state = Ot * torch.tanh(cell_state)

            if self.return_sequence_states :
                hidden_states[:, timestep, :] = hidden_state
                cell_states[:, timestep, :] = cell_state



        if self.return_sequence_states :
            return hidden_states, cell_states   # shape = (BxTxH)
        return hidden_state, cell_state         # shape = (BxH)

In [ ]:
class simpleLSTM(nn.Module) :

    def __init__(self, input_dim, hidden_dim, device, return_sequence_states = False) :
        super().__init__()
        self.hidden_dim = hidden_dim
        self.return_sequence_states = return_sequence_states
        self.device = device

        self.forget_gate = nn.Linear(
            in_features = input_dim + hidden_dim,
            out_features = hidden_dim 
        )
        
        self.input_gate_i = nn.Linear(
            in_features = input_dim + hidden_dim,
            out_features = hidden_dim 
        )

        self.input_gate_c = nn.Linear(
            in_features = input_dim + hidden_dim,
            out_features = hidden_dim 
        )

        self.output_gate = nn.Linear(
            in_features = input_dim + hidden_dim,
            out_features = hidden_dim 
        )


    def forward(self, embedded_batch) : 
        # input shape : (B, T, E)
        batch_size, timesteps_count, embedding_dim = embedded_batch.shape    

        hidden_state = torch.zeros(size=(batch_size, self.hidden_dim), device= self.device)
        cell_state = torch.zeros(size=(batch_size, self.hidden_dim), device= self.device)

        if self.return_sequence_states :
            hidden_states = torch.zeros(size=(batch_size, timesteps_count, self.hidden_dim), device= self.device)
            cell_states = torch.zeros(size=(batch_size, timesteps_count, self.hidden_dim), device= self.device)

        for timestep in range(timesteps_count):
            input = embedded_batch[: , timestep, :]    # shape = (B, E)
            X_t = torch.cat(tensors=(hidden_state,input ), dim=1)

            Ft = torch.sigmoid(self.forget_gate(X_t))
            cell_state = cell_state * Ft

            It = torch.sigmoid(self.input_gate_i(X_t))
            Canditate_t = torch.tanh(self.input_gate_c(X_t))
            cell_state = cell_state + (It * Canditate_t)

            Ot = torch.sigmoid(self.output_gate(X_t))
            hidden_state = Ot * torch.tanh(cell_state)

            if self.return_sequence_states :
                hidden_states[:, timestep, :] = hidden_state
                cell_states[:, timestep, :] = cell_state



        if self.return_sequence_states :
            return hidden_states, cell_states   # shape = (BxTxH)
        return hidden_state, cell_state         # shape = (BxH)

In [ ]:
class deepLSTM(nn.Module) :
    def __init__(self, layer_io_pairs, device, return_all_final_states = False):
        super().__init__()
        self.device = device 
        self.return_final_states = return_all_final_states
        self.layer_io_pair = layer_io_pairs
        self.module_list = nn.ModuleList()
        
        for i in range(len(self.layer_io_pair)) :

            in_dim, out_dim = self.layer_io_pair[i]

            if i != (len(self.layer_io_pair) -1) and out_dim != self.layer_io_pair[i+1][0] : #check layer compatibilty
                raise ValueError(f'NOT COMPATIBLE DIMENSIONS , for layers {i} and {i+1}')

            self.module_list.append(simpleLSTM(in_dim , out_dim, self.device, True))

    def forward(self, embedded_batch):
        # batch dim = BxTxE

        hidden_states = []
        cell_states = []

        for i in range(len(self.module_list)) : 
            layer = self.module_list[i]
            if i ==0 : 
                hidden_states, cell_states = layer(embedded_batch)
            else:
                hidden_states, cell_states = layer(hidden_states)

        if self.return_final_states :
            return hidden_states
        return hidden_states[:, -1, :]